In [5]:
import os

dataset1_path = r"C:\Users\ravis\Downloads\Traffic Light Management.v1i.yolov8"   # e.g., "./dataset1"
dataset2_path = r"C:\Users\ravis\Downloads\Traffic light management project.v2-traffic_flow_dataset.yolov8"  # e.g., "./roboflow_dataset2"

In [7]:
import shutil
from tqdm import tqdm

combined_temp = "all_data_combined"
os.makedirs(f"{combined_temp}/images", exist_ok=True)
os.makedirs(f"{combined_temp}/labels", exist_ok=True)

def copy_train_images_labels(src_dataset, dataset_name):
    src_images = os.path.join(src_dataset, "train", "images")
    src_labels = os.path.join(src_dataset, "train", "labels")
    
    # Get all image files (adjust extensions if needed)
    images = [f for f in os.listdir(src_images) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    for img_file in tqdm(images, desc=f"Copying {dataset_name}"):
        # Copy image
        shutil.copy(os.path.join(src_images, img_file),
                    os.path.join(combined_temp, "images", img_file))
        
        # Copy corresponding label file (same name but .txt)
        label_file = os.path.splitext(img_file)[0] + ".txt"
        src_label = os.path.join(src_labels, label_file)
        if os.path.exists(src_label):
            shutil.copy(src_label, os.path.join(combined_temp, "labels", label_file))
        else:
            print(f"Warning: No label for {img_file}")

# Copy from both datasets
copy_train_images_labels(dataset1_path, "dataset1")
copy_train_images_labels(dataset2_path, "dataset2")

Copying dataset2: 100%|█████████████████████████████████████████████████████████████| 747/747 [00:01<00:00, 507.85it/s]


In [9]:
import random
from sklearn.model_selection import train_test_split

# Get all image base names (without extension)
image_files = [f for f in os.listdir(f"{combined_temp}/images") 
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
image_names = [os.path.splitext(f)[0] for f in image_files]

# Split into train and validation sets
train_names, val_names = train_test_split(image_names, test_size=0.2, random_state=42)

print(f"Total images: {len(image_names)}")
print(f"Train: {len(train_names)} images")
print(f"Val: {len(val_names)} images")

Total images: 1128
Train: 902 images
Val: 226 images


In [11]:
final_dataset = "traffic_light_dataset"   # you can rename this
os.makedirs(f"{final_dataset}/train/images", exist_ok=True)
os.makedirs(f"{final_dataset}/train/labels", exist_ok=True)
os.makedirs(f"{final_dataset}/val/images", exist_ok=True)
os.makedirs(f"{final_dataset}/val/labels", exist_ok=True)

def move_files(name_list, subset):
    for name in tqdm(name_list, desc=f"Moving to {subset}"):
        # Find image file (with any extension)
        img_found = False
        for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
            src_img = os.path.join(combined_temp, "images", name + ext)
            if os.path.exists(src_img):
                shutil.move(src_img, os.path.join(final_dataset, subset, "images", name + ext))
                img_found = True
                break
        if not img_found:
            print(f"Warning: Image for {name} not found")
        
        # Move label
        src_label = os.path.join(combined_temp, "labels", name + ".txt")
        if os.path.exists(src_label):
            shutil.move(src_label, os.path.join(final_dataset, subset, "labels", name + ".txt"))

move_files(train_names, "train")
move_files(val_names, "val")

Moving to val: 100%|████████████████████████████████████████████████████████████████| 226/226 [00:00<00:00, 946.85it/s]


In [13]:
import yaml

# Use absolute paths for train/val (recommended)
train_path = os.path.abspath(f"{final_dataset}/train/images")
val_path = os.path.abspath(f"{final_dataset}/val/images")

data = {
    'train': train_path,
    'val': val_path,
    'nc': 5,
    'names': ['Auto', 'Bike', 'Bus', 'Car', 'Truck']
}

with open(f"{final_dataset}/data.yaml", 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("data.yaml created at:", os.path.abspath(f"{final_dataset}/data.yaml"))

data.yaml created at: C:\Users\ravis\TLM\Final Project\traffic_light_dataset\data.yaml


In [5]:
import os 
final_dataset = "traffic_light_dataset"
print("Train images:", len(os.listdir(f"{final_dataset}/train/images")))
print("Train labels:", len(os.listdir(f"{final_dataset}/train/labels")))
print("Val images:", len(os.listdir(f"{final_dataset}/val/images")))
print("Val labels:", len(os.listdir(f"{final_dataset}/val/labels")))

Train images: 902
Train labels: 902
Val images: 226
Val labels: 226


In [17]:
# Look at first label file in train
import glob
label_files = glob.glob(f"{final_dataset}/train/labels/*.txt"
if label_files:
    with open(label_files[0], 'r') as f:
        print(f"Sample from {os.path.basename(label_files[0])}:\n{f.read()}")

Sample from frame_0003_jpg.rf.6b6dfc7967877c10d4d1d28c74d9b231.txt:
3 0.85 0.83046875 0.25 0.3390625
1 0.9375 0.8125 0.05 0.18046875
3 0.6078125 0.48984375 0.14296875 0.28125
0 0.44375 0.38828125 0.08359375 0.2484375
0 0.38671875 0.27734375 0.0453125 0.15078125
3 0.4109375 0.1 0.01796875 0.04296875


In [19]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # or yolov8s.pt, etc.
model.train(data=f"{final_dataset}/data.yaml", epochs=50, imgsz=640, batch=16)

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.163  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=traffic_light_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, o

100%|███████████████████████████████████████████████████████████████████████████████| 755k/755k [00:00<00:00, 5.39MB/s]

Overriding model.yaml nc=80 with nc=5

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    164608  ultralytics.nn.modules.block.SPPF            [256, 256, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 11             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 12                  -1  1    148224  ultralytics.nn.modules.block.C2f             [384, 128, 1]                 
 13                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
 14             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
 15                  -1  1     37248  ultralytics.nn.modules.block.C2f             [192, 64, 1]                  
 16                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 

100%|█████████████████████████████████████████████████████████████████████████████| 5.35M/5.35M [00:01<00:00, 4.14MB/s]


...

                   all        226       2082      0.868      0.736      0.807      0.531



50 epochs completed in 0.148 hours.
Optimizer stripped from runs\detect\train\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train\weights\best.pt, 6.2MB

Validating runs\detect\train\weights\best.pt...
Ultralytics 8.3.163  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<0


                   all        226       2082      0.897      0.714      0.818      0.543
                  Auto        169        464      0.916      0.754      0.856      0.537
                  Bike        209        937      0.849       0.62      0.728      0.319
                   Bus         24         27      0.921      0.815      0.949      0.741
                   Car        173        644      0.881      0.783      0.885      0.603
                 Truck          9         10      0.917        0.6       0.67      0.515
Speed: 0.2ms preprocess, 1.9ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to runs\detect\train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000014D436230E0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [21]:
from ultralytics import YOLO

# Load your trained model (the best weights are saved in runs/detect/train/weights/best.pt)
model = YOLO('runs/detect/train/weights/best.pt')

# Validate on your validation set
metrics = model.val()  # automatically uses the data.yaml from training

# Print metrics
print(metrics.box.map)    # mAP50-95
print(metrics.box.map50)  # mAP50
print(metrics.box.mp)     # mean precision
print(metrics.box.mr)     # mean recall

Ultralytics 8.3.163  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 494.3180.5 MB/s, size: 102.2 KB)


val: Scanning C:\Users\ravis\TLM\Final Project\traffic_light_dataset\val\labels.cache... 226 images, 2 backgrounds, 0 c
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 15/15 [00:02


                   all        226       2082        0.9      0.723      0.821      0.544
                  Auto        169        464      0.914      0.754       0.86      0.539
                  Bike        209        937      0.848      0.624      0.732      0.319
                   Bus         24         27      0.958      0.849      0.957      0.743
                   Car        173        644      0.877      0.785      0.885      0.602
                 Truck          9         10      0.905        0.6      0.669      0.515
Speed: 1.0ms preprocess, 4.7ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to runs\detect\val
0.5436257778596015
0.8207085012644504
0.900381482845296
0.7225821622312625


In [25]:
from ultralytics import YOLO

# Load YOLOv8s
model = YOLO('yolov8s.pt')

# Train with maybe more epochs and early stopping
model.train(data='traffic_light_dataset/data.yaml',
            epochs=100,
            imgsz=640,
            batch=16,
            patience=20,
            name='yolov8s_run')

100%|█████████████████████████████████████████████████████████████████████████████| 21.5M/21.5M [00:04<00:00, 4.74MB/s]


New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.163  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=traffic_light_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_run, nbs=64, nms=F

train: Scanning C:\Users\ravis\TLM\Final Project\traffic_light_dataset\train\labels.cache... 902 images, 0 backgrounds,

WARNING Box and segment counts should be equal, but got len(segments) = 107, len(boxes) = 7997. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.


val: Fast image access  (ping: 0.00.0 ms, read: 765.7299.8 MB/s, size: 97.8 KB)


...

                   all        226       2082      0.848      0.735       0.82      0.524
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 58, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



78 epochs completed in 0.379 hours.
Optimizer stripped from runs\detect\yolov8s_run\weights\last.pt, 22.5MB
Optimizer stripped from runs\detect\yolov8s_run\weights\best.pt, 22.5MB

Validating runs\detect\yolov8s_run\weights\best.pt...
Ultralytics 8.3.163  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 8/8 [00:02<0


                   all        226       2082      0.846      0.821      0.869      0.549
                  Auto        169        464        0.9      0.835      0.882      0.557
                  Bike        209        937      0.827      0.723      0.785      0.336
                   Bus         24         27      0.861      0.889      0.927      0.707
                   Car        173        644      0.843      0.869       0.89      0.605
                 Truck          9         10      0.797      0.787       0.86      0.541
Speed: 0.3ms preprocess, 4.3ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to runs\detect\yolov8s_run


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000014C755997C0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

In [1]:
import torch
torch.cuda.empty_cache()

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Now import everything else
from ultralytics import YOLO
import torch

In [1]:
from ultralytics import YOLO

# Load YOLOv8m pretrained model
model = YOLO('yolov8m.pt')

# Train with memory-optimized settings for 6GB GPU
results = model.train(
    data='traffic_light_dataset/data.yaml',  # your dataset path
    epochs=100,                                # enough for convergence
    imgsz=640,                                 # standard size (smaller = less memory)
    batch=8,                                   # conservative for 6GB + yolov8m
    amp=True,                                  # CRITICAL: mixed precision halves memory usage
    patience=20,                               # early stopping if no improvement
    workers=4,                                 # reduce from default 8 to save memory
    device=0,                                  # use GPU
    name='yolov8m_traffic_final'               # experiment name
)

Ultralytics 8.4.33 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=traffic_light_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_traffic_final14, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

In [4]:
import os
print(os.getcwd())

/home/ravis/projects/TLM/Final Project


In [5]:
print(os.listdir())

['.ipynb_checkpoints', 'yolov8s.pt', 'yolov8n.pt', 'Untitled1.ipynb', 'runs', 'yolo11n.pt', 'all_data_combined', 'yolov8m.pt', 'traffic_light_dataset']
